# Cu FEFF: portable 128D encoder reruns + optional 64D comparison

This notebook does not require my local checkpoint state.

What it does:

1. Uses any existing 128D encoder runs it finds under `output/training/gnnXASStyleCuFEFF128D`.
2. If a requested encoder seed is missing, it can train it automatically.
3. Trains/fetches ExpertXAS heads for the selected 128D encoders.
4. If existing 64D D1 ensemble files are present, it compares 128D against 64D and mixed 64D+128D ensembles.
5. If 64D files are not present, it still runs the 128D workflow and skips 64D-only rows.

No 96D path remains.


In [ ]:
from pathlib import Path
import glob, json, os, random, re
from datetime import datetime

import dgl
import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from pymatgen.core import Lattice, Structure
from torch import nn
from torch.utils.data import DataLoader, Dataset

from matgl.config import DEFAULT_ELEMENTS
try:
    from matgl.graph.compute import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
except Exception:
    from matgl.graph._compute_dgl import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
from matgl.ext.pymatgen import Structure2Graph
from matgl.layers import MLP as M3GNetMLP, ActivationFunction, BondExpansion, EmbeddingBlock, GatedMLP, M3GNetBlock, SphericalBesselWithHarmonics, ThreeBodyInteractions
from matgl.utils.cutoff import polynomial_cutoff

import matgl.layers._basis as matgl_basis
import matgl.layers._three_body as matgl_three_body
import matgl.utils.maths as matgl_math

from omnixas.data import MLData, MLSplits
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "omnixas").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Run this notebook from inside the OmniXAS repo")
    REPO_ROOT = REPO_ROOT.parent


def patch_matgl_gpu_constants():
    def safe_sbf(self, r):
        cutoff = torch.as_tensor(self.cutoff, dtype=r.dtype, device=r.device)
        roots = matgl_basis.SPHERICAL_BESSEL_ROOTS[: self.max_l, : self.max_n].to(r.device, r.dtype)
        r_c = r.clamp(max=cutoff)
        factor = torch.sqrt(torch.as_tensor(2.0, dtype=r.dtype, device=r.device) / cutoff**3)
        return torch.cat([
            self.funcs[i](r_c[:, None] * roots[i][None, :] / cutoff) * factor / torch.abs(self.funcs[i + 1](roots[i][None, :]))
            for i in range(self.max_l)
        ], axis=1)

    def safe_combine(sbf, shf, max_n, max_l, use_phi):
        if sbf.size(0) == 0:
            return sbf
        if use_phi:
            repeats = torch.repeat_interleave(2 * torch.arange(max_l, device=sbf.device) + 1, max_n)
            blocks = 2 * torch.arange(max_l, device=sbf.device) + 1
        else:
            repeats = torch.ones(max_l * max_n, dtype=torch.long, device=sbf.device)
            blocks = [1] * max_l
        expanded_sbf = torch.repeat_interleave(sbf, repeats, 1)
        col, idx, start = torch.arange(shf.size(1), device=shf.device), [], 0
        for block in blocks:
            block = int(block.item()) if torch.is_tensor(block) else int(block)
            idx.append(torch.tile(col[start:start + block], [max_n]))
            start += block
        expanded_shf = torch.index_select(shf, 1, torch.cat(idx))
        return torch.reshape(expanded_sbf * expanded_shf, [-1, max_n * max_l * (max_l if use_phi else 1)])

    def safe_scatter_sum(x, segment_ids, num_segments, dim):
        segment_ids = matgl_math.broadcast(segment_ids.to(x.device), x, dim)
        size = list(x.size())
        size[dim] = 0 if segment_ids.numel() == 0 else num_segments
        return torch.zeros(size, dtype=x.dtype, device=x.device).scatter_add_(dim, segment_ids, x)

    matgl_basis.SphericalBesselFunction._call_sbf = safe_sbf
    matgl_basis.combine_sbf_shf = safe_combine
    matgl_three_body.combine_sbf_shf = safe_combine
    matgl_math.scatter_sum = safe_scatter_sum
    matgl_three_body.scatter_sum = safe_scatter_sum


patch_matgl_gpu_constants()
torch.set_float32_matmul_precision("medium")


In [ ]:
TASK = "Cu_FEFF"
ENCODER_DIM = 128

# Requested 128D seeds. Existing runs are reused; missing runs can be trained automatically.
KEPT_ENCODER_SEEDS = [45]
NEW_ENCODER_SEEDS = [52, 53, 54, 55, 56]
ENCODER_SEEDS = KEPT_ENCODER_SEEDS + NEW_ENCODER_SEEDS

# Switches:
# - TRAIN_NEW_ENCODERS=True retrains NEW_ENCODER_SEEDS even if runs already exist.
# - AUTO_TRAIN_MISSING_ENCODERS=True makes the notebook portable on a new machine.
# - TRAIN_MLPS=True trains heads for newly trained encoders; existing heads are reused when present.
# - INCLUDE_EXISTING_MLPS=True includes already-trained heads in the comparison.
TRAIN_NEW_ENCODERS = True
AUTO_TRAIN_MISSING_ENCODERS = True
TRAIN_MLPS = True
INCLUDE_EXISTING_MLPS = True

RAW_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")) / "materialscloud_omnixas_raw" / "extracted"
DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
ID_SITE_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"

RUN_ROOT = REPO_ROOT / "output" / "training" / "gnnXASStyleCuFEFF128D"
COMPARISON_DIR = RUN_ROOT / "comparisons" / f"{datetime.now():%Y%m%d_%H%M%S}"
MLP_RUN_NAME = f"mlp_{datetime.now():%Y%m%d_%H%M%S}"

FEATURE_SCALE = 1000.0
BATCH_SIZE = 32
NUM_WORKERS = 4
ACCELERATOR = "gpu"

GNN_CUTOFF = 4.0
GNN_THREEBODY_CUTOFF = 4.0
GNN_BLOCKS = 3
GNN_LR = 5e-4
GNN_EPOCHS = 300
USE_ENCODER_EARLY_STOPPING = False
GNN_PATIENCE = 60  # only used if USE_ENCODER_EARLY_STOPPING=True
GNN_WARMUP_EPOCHS = 10
ENCODER_EXPORT_CHECKPOINT = "last"  # final annealed weights by default
GNN_DROPOUT = 0.10
GNN_WEIGHT_DECAY = 1e-4
TEMP_HEAD_DIMS = [128, 128]
TEMP_HEAD_DROPOUT = 0.25

EXPERT_DIMS = [600, 600, 400]
EXPERT_MAX_EPOCHS = 800
EXPERT_PATIENCE = 60
EXPERT_BATCH_SIZE = 32

# Four heads per encoder is enough; head diversity saturates quickly.
EXPERT_CONFIGS = [
    {"seed": 42, "dropout": 0.50, "lr": 1e-3},
    {"seed": 43, "dropout": 0.40, "lr": 1e-3},
    {"seed": 44, "dropout": 0.30, "lr": 1e-3},
    {"seed": 45, "dropout": 0.25, "lr": 7e-4},
]

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("repo:", REPO_ROOT)
print("device:", DEVICE)
print("requested 128D encoder seeds:", ENCODER_SEEDS)
print("TRAIN_NEW_ENCODERS:", TRAIN_NEW_ENCODERS, "AUTO_TRAIN_MISSING_ENCODERS:", AUTO_TRAIN_MISSING_ENCODERS)
print("TRAIN_MLPS:", TRAIN_MLPS, "INCLUDE_EXISTING_MLPS:", INCLUDE_EXISTING_MLPS)
print("encoder training:", {"lr": GNN_LR, "epochs": GNN_EPOCHS, "early_stop": USE_ENCODER_EARLY_STOPPING, "warmup": GNN_WARMUP_EPOCHS, "export": ENCODER_EXPORT_CHECKPOINT, "wd": GNN_WEIGHT_DECAY})


In [ ]:
def seed_all(seed):
    pl.seed_everything(seed, workers=True)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)


def eta(pred, target, train_y):
    baseline = np.repeat(train_y.mean(axis=0, keepdims=True), len(target), axis=0)
    median_mse = float(np.median(np.mean((target - pred) ** 2, axis=1)))
    baseline_median_mse = float(np.median(np.mean((target - baseline) ** 2, axis=1)))
    return {"eta": baseline_median_mse / median_mse, "median_mse": median_mse, "baseline_median_mse": baseline_median_mse}


def load_xasblock(path, input_dim):
    state = torch.load(path, map_location="cpu")["state_dict"]
    weights = {k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")}
    if not weights:
        weights = {k.removeprefix("model.model."): v for k, v in state.items() if k.startswith("model.model.")}
    model = XASBlock(input_dim, EXPERT_DIMS, 141)
    model.load_state_dict(weights)
    return model.eval()


@torch.no_grad()
def predict_array(model, X, batch_size=1024):
    model = model.to(DEVICE).eval()
    preds = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds)


def read_id_site(split):
    return [(mid, int(site)) for mid, site in (line.rsplit("_", 1) for line in (ID_SITE_DIR / f"{TASK}_{split}.txt").read_text().splitlines() if line.strip())]


def parse_feff_structure(path):
    abc = angles = None
    species, coords = [], []
    site_re = re.compile(r"^\*\s+\d+\s+([A-Z][a-z]?)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)")
    for line in path.read_text(errors="ignore").splitlines():
        if line.startswith("TITLE abc:"):
            abc = [float(x) for x in line.split(":", 1)[1].split()[:3]]
        elif line.startswith("TITLE angles:"):
            angles = [float(x) for x in line.split(":", 1)[1].split()[:3]]
        elif (match := site_re.match(line)):
            species.append(match.group(1))
            coords.append([float(match.group(2)), float(match.group(3)), float(match.group(4))])
    if abc is None or angles is None or not species:
        raise ValueError(f"Could not parse FEFF structure: {path}")
    return Structure(Lattice.from_parameters(*abc, *angles), species, coords, coords_are_cartesian=False)


def load_cu_feff_structure(material_id, site):
    material_dir = RAW_ROOT / "FEFF" / "Cu" / material_id
    poscar = material_dir / "POSCAR"
    if poscar.exists():
        return Structure.from_file(poscar)
    return parse_feff_structure(material_dir / "FEFF-XANES" / f"{site:03d}_Cu" / "feff.inp")


In [ ]:
class CuFEFFDataset(Dataset):
    def __init__(self, split):
        self.ids = read_id_site(split)
        self.y = np.loadtxt(DATA_DIR / f"{TASK}_{split}_y.txt", dtype=np.float32)
        self.cache = {}

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        material_id, site = self.ids[idx]
        if material_id not in self.cache:
            self.cache[material_id] = load_cu_feff_structure(material_id, site)
        return self.cache[material_id], site, torch.tensor(self.y[idx])


class GraphBatcher:
    def __init__(self, gnn):
        self.converter = Structure2Graph(gnn.element_types, gnn.cutoff)

    def graph(self, structure):
        out = self.converter.get_graph(structure)
        graph = out[0]
        lat_src = structure.lattice.matrix if len(out) == 2 else (out[1][0] if getattr(out[1], "ndim", 0) == 3 else out[1])
        lat = torch.tensor(np.array(lat_src, copy=True), dtype=torch.float32)

        if "pbc_offshift" in graph.edata:
            graph.edata["pbc_offshift"] = graph.edata["pbc_offshift"].float()
        else:
            graph.edata["pbc_offshift"] = graph.edata["pbc_offset"].float() @ lat

        if "pos" in graph.ndata:
            graph.ndata["pos"] = graph.ndata["pos"].float()
        elif "frac_coords" in graph.ndata:
            graph.ndata["pos"] = graph.ndata["frac_coords"].float() @ lat
        else:
            graph.ndata["pos"] = torch.tensor(np.array(structure.frac_coords, copy=True), dtype=torch.float32) @ lat

        graph.edata["bond_vec"], graph.edata["bond_dist"] = compute_pair_vector_and_distance(graph)
        return graph

    def __call__(self, batch):
        graphs, sites, ys, offset = [], [], [], 0
        for structure, site, y in batch:
            graph = self.graph(structure)
            graphs.append(graph)
            sites.append(offset + site)
            ys.append(y)
            offset += graph.num_nodes()
        return {"graph": dgl.batch(graphs), "site": torch.tensor(sites), "y": torch.stack(ys).float()}


In [ ]:
class XASGNN(nn.Module):
    def __init__(self, encoder_dim=ENCODER_DIM):
        super().__init__()
        act = ActivationFunction["swish"].value()
        max_n = max_l = 3
        degree = max_n * max_l
        self.element_types = DEFAULT_ELEMENTS
        self.cutoff = GNN_CUTOFF
        self.threebody_cutoff = GNN_THREEBODY_CUTOFF
        self.bond_expansion = BondExpansion(max_l, max_n, GNN_CUTOFF)
        self.basis_expansion = SphericalBesselWithHarmonics(max_n, max_l, GNN_CUTOFF, use_smooth=False, use_phi=False)
        self.embedding = EmbeddingBlock(degree_rbf=degree, dim_node_embedding=encoder_dim, dim_edge_embedding=encoder_dim, ntypes_node=len(DEFAULT_ELEMENTS), activation=act)
        self.three_body_interactions = nn.ModuleList([
            ThreeBodyInteractions(
                update_network_atom=M3GNetMLP(dims=[encoder_dim, degree], activation=nn.Sigmoid(), activate_last=True),
                update_network_bond=GatedMLP(in_feats=degree, dims=[encoder_dim], use_bias=False),
            ) for _ in range(GNN_BLOCKS)
        ])
        self.graph_layers = nn.ModuleList([
            M3GNetBlock(degree=degree, activation=act, conv_hiddens=[encoder_dim, encoder_dim], dim_node_feats=encoder_dim, dim_edge_feats=encoder_dim, dropout=GNN_DROPOUT)
            for _ in range(GNN_BLOCKS)
        ])

    def forward(self, graph, site):
        graph.edata["rbf"] = self.bond_expansion(graph.edata["bond_dist"])
        line_graph = create_line_graph(graph.to("cpu"), self.threebody_cutoff).to(graph.device)
        line_graph.apply_edges(compute_theta_and_phi)
        basis = self.basis_expansion(line_graph)
        cutoff = polynomial_cutoff(graph.edata["bond_dist"], self.threebody_cutoff)
        node, edge, state = self.embedding(graph.ndata["node_type"], graph.edata["rbf"], None)
        for i in range(GNN_BLOCKS):
            edge = self.three_body_interactions[i](graph, line_graph, basis, cutoff, node, edge)
            edge, node, state = self.graph_layers[i](graph, edge, node, state)
        return node[site]


class SpectrumHead(nn.Module):
    def __init__(self, encoder_dim=ENCODER_DIM):
        super().__init__()
        dims = [encoder_dim, *TEMP_HEAD_DIMS, 141]
        layers = []
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(a, b))
            if i < len(dims) - 2:
                layers += [nn.LayerNorm(b), nn.SiLU(), nn.Dropout(TEMP_HEAD_DROPOUT)]
            else:
                layers.append(nn.Softplus())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class XASGNNModel(nn.Module):
    def __init__(self, encoder_dim=ENCODER_DIM):
        super().__init__()
        self.gnn = XASGNN(encoder_dim)
        self.head = SpectrumHead(encoder_dim)

    def forward(self, graph, site):
        z = self.gnn(graph, site)
        return self.head(z * FEATURE_SCALE), z


class LitGNN(pl.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.val_mses = []

    def step(self, batch, split):
        pred, _ = self.model(batch["graph"].to(self.device), batch["site"].to(self.device))
        y = batch["y"].to(self.device)
        loss = ((pred - y) ** 2).mean()
        self.log(f"{split}_loss", loss, on_epoch=True, prog_bar=True)
        if split == "val":
            self.val_mses.append(((pred - y) ** 2).mean(dim=1).detach())
        return loss

    def training_step(self, batch, _):
        return self.step(batch, "train")

    def on_validation_epoch_start(self):
        self.val_mses = []

    def validation_step(self, batch, _):
        return self.step(batch, "val")

    def on_validation_epoch_end(self):
        if self.val_mses:
            self.log("val_median_mse", torch.cat(self.val_mses).median(), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW([
            {"params": self.model.gnn.parameters(), "lr": GNN_LR},
            {"params": self.model.head.parameters(), "lr": 1e-3},
        ], weight_decay=GNN_WEIGHT_DECAY)
        warmup = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=GNN_WARMUP_EPOCHS)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(1, GNN_EPOCHS - GNN_WARMUP_EPOCHS), eta_min=1e-6)
        scheduler = torch.optim.lr_scheduler.SequentialLR(opt, [warmup, cosine], milestones=[GNN_WARMUP_EPOCHS])
        return {"optimizer": opt, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}


In [ ]:
def train_encoder(seed, run_dir):
    seed_all(seed)
    model = XASGNNModel(ENCODER_DIM)
    batcher = GraphBatcher(model.gnn)
    train_loader = DataLoader(CuFEFFDataset("train"), BATCH_SIZE, shuffle=True, collate_fn=batcher, num_workers=NUM_WORKERS)
    val_loader = DataLoader(CuFEFFDataset("val"), BATCH_SIZE, shuffle=False, collate_fn=batcher, num_workers=NUM_WORKERS)

    print(f"\n=== train 128D encoder seed {seed} ===")
    print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
    ckpt = ModelCheckpoint(run_dir / "gnn_checkpoints", filename="best-{epoch:03d}-{val_median_mse:.5f}", monitor="val_median_mse", mode="min", save_top_k=1, save_last=True)
    callbacks = [ckpt]
    if USE_ENCODER_EARLY_STOPPING:
        callbacks.insert(0, EarlyStopping(monitor="val_median_mse", patience=GNN_PATIENCE, mode="min"))
    trainer = pl.Trainer(
        max_epochs=GNN_EPOCHS,
        accelerator=ACCELERATOR,
        devices=1,
        callbacks=callbacks,
        logger=CSVLogger(str(run_dir), name="gnn_logs"),
        log_every_n_steps=1,
    )
    trainer.fit(LitGNN(model), train_loader, val_loader)
    if ENCODER_EXPORT_CHECKPOINT == "last":
        last = run_dir / "gnn_checkpoints" / "last.ckpt"
        if not last.exists():
            raise RuntimeError(f"No last encoder checkpoint saved for {run_dir}")
        return last
    if not ckpt.best_model_path:
        raise RuntimeError(f"No best encoder checkpoint saved for {run_dir}")
    return Path(ckpt.best_model_path)


def latest_run_for_seed(seed):
    root = RUN_ROOT.resolve()
    candidates = sorted(
        [
            run for run in root.rglob(f"*_seed{seed}")
            if run.is_dir() and list((run / "gnn_checkpoints").glob("best*.ckpt"))
        ],
        key=lambda run: run.stat().st_mtime,
    )
    return candidates[-1] if candidates else None


def best_encoder_ckpt(run_dir):
    ckpts = sorted((run_dir / "gnn_checkpoints").glob("best*.ckpt"))
    if not ckpts:
        raise FileNotFoundError(f"No best encoder checkpoint in {run_dir}")
    return ckpts[0]


def load_encoder(ckpt_path):
    model = XASGNNModel(ENCODER_DIM)
    state = torch.load(ckpt_path, map_location="cpu")["state_dict"]
    model.gnn.load_state_dict({k.removeprefix("model.gnn."): v for k, v in state.items() if k.startswith("model.gnn.")}, strict=True)
    return model.eval()


@torch.no_grad()
def export_features(split, model):
    loader = DataLoader(CuFEFFDataset(split), BATCH_SIZE, shuffle=False, collate_fn=GraphBatcher(model.gnn), num_workers=NUM_WORKERS)
    model = model.to(DEVICE).eval()
    X, y = [], []
    for batch in loader:
        z = model.gnn(batch["graph"].to(DEVICE), batch["site"].to(DEVICE))
        X.append((z * FEATURE_SCALE).cpu().numpy())
        y.append(batch["y"].numpy())
    return np.concatenate(X), np.concatenate(y)


def load_or_export_features(run_dir, encoder):
    feature_dir = run_dir / "features"
    files = {split: feature_dir / f"{TASK}_{split}_X_128d.txt" for split in ["train", "val", "test"]}
    y_files = {split: feature_dir / f"{TASK}_{split}_y.txt" for split in ["train", "val", "test"]}
    if all(path.exists() for path in files.values()) and all(path.exists() for path in y_files.values()):
        X = {split: np.loadtxt(files[split], dtype=np.float32) for split in files}
        y = {split: np.loadtxt(y_files[split], dtype=np.float32) for split in y_files}
        return X, y

    feature_dir.mkdir(exist_ok=True)
    X, y = {}, {}
    for split in ["train", "val", "test"]:
        X[split], y[split] = export_features(split, encoder)
        print(split, X[split].shape, y[split].shape)

    for split in ["train", "val", "test"]:
        np.savetxt(files[split], X[split])
        np.savetxt(y_files[split], y[split])
    return X, y


def train_expert_head(cfg, split_data, run_dir):
    seed_all(cfg["seed"])
    XASBlock.DROPOUT = cfg["dropout"]
    out_dir = run_dir / "expert_heads" / f"{MLP_RUN_NAME}_seed{cfg['seed']}_dropout{str(cfg['dropout']).replace('.', 'p')}_lr{cfg['lr']:g}"
    reg = XASBlockRegressor(
        directory=str(out_dir),
        input_dim=ENCODER_DIM,
        hidden_dims=EXPERT_DIMS,
        output_dim=141,
        initial_lr=cfg["lr"],
        batch_size=EXPERT_BATCH_SIZE,
        max_epochs=EXPERT_MAX_EPOCHS,
        early_stopping_patience=EXPERT_PATIENCE,
        use_lr_finder=False,
        monitor_metric="val_median_mse",
        shuffle=True,
        lr_scheduler="cosine",
        cosine_t_max=EXPERT_MAX_EPOCHS,
        cosine_eta_min=1e-6,
    )
    reg.fit(split_data).load("best")
    return Path(reg.cfg.fetch_checkpoint("best"))


def existing_head_ckpts(run_dir):
    return sorted((run_dir / "expert_heads").glob("*/best*.ckpt"))


In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

train_y_ref = np.loadtxt(DATA_DIR / f"{TASK}_train_y.txt", dtype=np.float32)
y_ref = {s: np.loadtxt(DATA_DIR / f"{TASK}_{s}_y.txt", dtype=np.float32) for s in ["val", "test"]}

all_members = []
all_preds_val = []
all_preds_test = []

for seed in ENCODER_SEEDS:
    existing_run = latest_run_for_seed(seed)
    should_train_encoder = (seed in NEW_ENCODER_SEEDS and TRAIN_NEW_ENCODERS) or (existing_run is None and AUTO_TRAIN_MISSING_ENCODERS)

    if should_train_encoder:
        run_dir = RUN_ROOT / f"{datetime.now():%Y%m%d_%H%M%S}_seed{seed}"
        run_dir.mkdir(parents=True, exist_ok=False)
        encoder_ckpt = train_encoder(seed, run_dir)
    elif existing_run is not None:
        run_dir = existing_run
        encoder_ckpt = best_encoder_ckpt(run_dir) if ENCODER_EXPORT_CHECKPOINT == "best" else (run_dir / "gnn_checkpoints" / "last.ckpt")
        if not encoder_ckpt.exists():
            encoder_ckpt = best_encoder_ckpt(run_dir)
    else:
        raise FileNotFoundError(f"No existing run for seed {seed}; set AUTO_TRAIN_MISSING_ENCODERS=True or remove this seed.")

    print(f"\n=== encoder seed {seed}: {run_dir} ===")
    print("encoder checkpoint:", encoder_ckpt)
    (run_dir / "run_metadata_latest_notebook.json").write_text(json.dumps({
        "task": TASK,
        "encoder_dim": ENCODER_DIM,
        "encoder_seed": seed,
        "trained_encoder_this_run": should_train_encoder,
        "auto_train_missing_encoders": AUTO_TRAIN_MISSING_ENCODERS,
        "train_mlps": TRAIN_MLPS,
        "include_existing_mlps": INCLUDE_EXISTING_MLPS,
        "expert_configs": EXPERT_CONFIGS,
        "mlp_run_name": MLP_RUN_NAME,
        "gnn_lr": GNN_LR,
        "gnn_epochs": GNN_EPOCHS,
        "gnn_early_stop": USE_ENCODER_EARLY_STOPPING,
        "gnn_warmup_epochs": GNN_WARMUP_EPOCHS,
        "encoder_export_checkpoint": ENCODER_EXPORT_CHECKPOINT,
        "gnn_dropout": GNN_DROPOUT,
        "gnn_weight_decay": GNN_WEIGHT_DECAY,
    }, indent=2))

    encoder = load_encoder(encoder_ckpt)
    X, y = load_or_export_features(run_dir, encoder)
    for split in ["val", "test"]:
        if not np.allclose(y[split], y_ref[split], atol=1e-5):
            raise RuntimeError(f"Exported {split} targets do not match ml_data for seed {seed}")

    split_data = MLSplits(
        train=MLData(X=X["train"], y=y["train"]),
        val=MLData(X=X["val"], y=y["val"]),
        test=MLData(X=X["test"], y=y["test"]),
    )

    head_ckpts = []
    if INCLUDE_EXISTING_MLPS:
        head_ckpts.extend(existing_head_ckpts(run_dir))
    if TRAIN_MLPS and (should_train_encoder or not head_ckpts):
        for cfg in EXPERT_CONFIGS:
            head_ckpts.append(train_expert_head(cfg, split_data, run_dir))
    if not head_ckpts:
        raise FileNotFoundError(f"No MLP heads selected for {run_dir}. Enable TRAIN_MLPS or INCLUDE_EXISTING_MLPS.")

    run_rows, run_pred_val, run_pred_test = [], [], []
    for head_ckpt in sorted(set(head_ckpts)):
        head = load_xasblock(head_ckpt, input_dim=ENCODER_DIM)
        pred_val = predict_array(head, X["val"])
        pred_test = predict_array(head, X["test"])
        val_metrics = eta(pred_val, y["val"], y["train"])
        test_metrics = eta(pred_test, y["test"], y["train"])
        row = {
            "family": "new_128d",
            "result_type": "single_head",
            "encoder_dim": ENCODER_DIM,
            "encoder_seed": seed,
            "head": head_ckpt.parent.name,
            "name": f"128D seed{seed} {head_ckpt.parent.name}",
            "n": 1,
            "val_eta": val_metrics["eta"],
            "test_eta": test_metrics["eta"],
            "val_median_mse": val_metrics["median_mse"],
            "test_median_mse": test_metrics["median_mse"],
            "path": str(head_ckpt),
        }
        all_members.append(row)
        run_rows.append(row)
        all_preds_val.append(pred_val)
        all_preds_test.append(pred_test)
        run_pred_val.append(pred_val)
        run_pred_test.append(pred_test)
        print(f"{row['name']}: val_eta={row['val_eta']:.3f} test_eta={row['test_eta']:.3f}")

    pd.DataFrame(run_rows).to_csv(run_dir / f"members_{MLP_RUN_NAME}.csv", index=False)
    np.savez_compressed(run_dir / f"predictions_{MLP_RUN_NAME}.npz", preds_val=np.stack(run_pred_val), preds_test=np.stack(run_pred_test), head=np.array([r["head"] for r in run_rows]))
    torch.cuda.empty_cache()

members_128d = pd.DataFrame(all_members).reset_index(drop=True)
members_128d.to_csv(COMPARISON_DIR / "members_128d.csv", index=False)
np.savez_compressed(COMPARISON_DIR / "predictions_128d.npz", preds_val=np.stack(all_preds_val), preds_test=np.stack(all_preds_test))
print("saved 128D members:", COMPARISON_DIR)


In [ ]:
def latest_64d_d1_summary():
    summaries = sorted((REPO_ROOT / "output" / "training" / "gnnXASStyleCuFEFF" / "d1_ensemble").glob("*/summary.json"))
    if not summaries:
        return None, None
    summary_path = summaries[-1]
    return summary_path, json.loads(summary_path.read_text())


comparison_rows = []
summary_path, baseline = latest_64d_d1_summary()
if baseline is not None:
    member_path = summary_path.parent / "members.csv"
    baseline_n = len(pd.read_csv(member_path)) if member_path.exists() else np.nan
    comparison_rows += [
        {
            "family": "existing_64d",
            "result_type": "single_head",
            "name": "existing 64D best single by val",
            "n": 1,
            "val_eta": baseline["best_single_val"],
            "test_eta": baseline["best_single_test"],
            "source": str(summary_path.parent),
        },
        {
            "family": "existing_64d",
            "result_type": "ensemble",
            "name": "existing 64D D1 global ensemble",
            "n": baseline_n,
            "val_eta": baseline["global_ensemble_val"],
            "test_eta": baseline["global_ensemble_test"],
            "source": str(summary_path.parent),
        },
    ]
else:
    print("No existing 64D D1 summary found; skipping 64D baseline rows.")

comparison_rows += members_128d.to_dict("records")

preds_val = np.stack(all_preds_val)
preds_test = np.stack(all_preds_test)


def ensemble_row(name, indices):
    val_metrics = eta(preds_val[indices].mean(axis=0), y_ref["val"], train_y_ref)
    test_metrics = eta(preds_test[indices].mean(axis=0), y_ref["test"], train_y_ref)
    return {
        "family": "new_128d",
        "result_type": "ensemble",
        "name": name,
        "n": len(indices),
        "val_eta": val_metrics["eta"],
        "test_eta": test_metrics["eta"],
        "val_median_mse": np.nan,
        "test_median_mse": np.nan,
    }


for seed, group in members_128d.groupby("encoder_seed"):
    comparison_rows.append(ensemble_row(f"128D seed{seed} ensemble", list(group.index)))

all_128d_indices = list(range(len(members_128d)))
comparison_rows.append(ensemble_row("128D all selected seeds ensemble", all_128d_indices))

# Optional mixed 64D+128D rows when 64D prediction files are available.
d1_prediction_files = sorted((REPO_ROOT / "output" / "training" / "gnnXASStyleCuFEFF" / "d1_ensemble").glob("*/member_predictions.npz"))
if d1_prediction_files:
    d1_predictions = np.load(d1_prediction_files[-1], allow_pickle=True)
    preds64_val = d1_predictions["preds_val"]
    preds64_test = d1_predictions["preds_test"]
    mean64_test = preds64_test.mean(axis=0)
    mean64_val = preds64_val.mean(axis=0)
    mean128_val, mean128_test = preds_val.mean(axis=0), preds_test.mean(axis=0)

    flat_mix_val = np.concatenate([preds64_val, preds_val], axis=0).mean(axis=0)
    flat_mix_test = np.concatenate([preds64_test, preds_test], axis=0).mean(axis=0)
    equal_mix_val = 0.5 * mean64_val + 0.5 * mean128_val
    equal_mix_test = 0.5 * mean64_test + 0.5 * mean128_test

    for name, pred_val, pred_test, n in [
        ("mixed flat 64D+128D ensemble", flat_mix_val, flat_mix_test, len(preds64_test) + len(preds_test)),
        ("mixed equal-family 64D+128D ensemble", equal_mix_val, equal_mix_test, 2),
    ]:
        val_metrics = eta(pred_val, y_ref["val"], train_y_ref)
        test_metrics = eta(pred_test, y_ref["test"], train_y_ref)
        comparison_rows.append({
            "family": "mixed_64d_128d",
            "result_type": "ensemble",
            "name": name,
            "n": n,
            "val_eta": val_metrics["eta"],
            "test_eta": test_metrics["eta"],
            "val_median_mse": np.nan,
            "test_median_mse": np.nan,
        })

    def bootstrap_eta_delta(pred_a, pred_b, target, train_y, n_boot=2000, seed=123):
        rng = np.random.default_rng(seed)
        n = len(target)
        baseline = np.repeat(train_y.mean(axis=0, keepdims=True), n, axis=0)
        base_mse = np.mean((target - baseline) ** 2, axis=1)
        mse_a = np.mean((target - pred_a) ** 2, axis=1)
        mse_b = np.mean((target - pred_b) ** 2, axis=1)
        deltas = np.empty(n_boot, dtype=np.float32)
        for i in range(n_boot):
            idx = rng.integers(0, n, n)
            base_med = np.median(base_mse[idx])
            deltas[i] = base_med / np.median(mse_a[idx]) - base_med / np.median(mse_b[idx])
        return float(np.quantile(deltas, 0.025)), float(np.quantile(deltas, 0.975))

    bootstrap_rows = []
    for name, pred in [
        ("128D selected ensemble minus 64D", mean128_test),
        ("mixed flat minus 64D", flat_mix_test),
        ("mixed equal-family minus 64D", equal_mix_test),
    ]:
        lo, hi = bootstrap_eta_delta(pred, mean64_test, y_ref["test"], train_y_ref)
        bootstrap_rows.append({"comparison": name, "delta_eta_ci95_low": lo, "delta_eta_ci95_high": hi})
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    bootstrap_df.to_csv(COMPARISON_DIR / "paired_bootstrap_test_delta_vs_64d.csv", index=False)
    display(bootstrap_df)
else:
    print("No 64D D1 member predictions found; skipping mixed ensemble and bootstrap rows.")

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(COMPARISON_DIR / "comparison_64d_vs_128d.csv", index=False)

cols = ["family", "result_type", "name", "n", "val_eta", "test_eta", "val_median_mse", "test_median_mse"]
shown = comparison[cols].sort_values(["result_type", "val_eta"], ascending=[True, False])
display(shown)
print("saved:", COMPARISON_DIR / "comparison_64d_vs_128d.csv")


## Test-MSE histogram: old baseline vs 64D ensemble vs best 128D

This plots per-spectrum test MSE for the old frozen-M3GNet ExpertXAS baseline, the existing 64D D1 ensemble, and the best 128D candidate selected by validation eta.


In [ ]:
import matplotlib.pyplot as plt


def latest_expert_ckpt():
    patterns = [
        REPO_ROOT / "output" / "training" / "expertXAS" / TASK / "runs" / "*" / "best*.ckpt",
        REPO_ROOT / "output" / "training" / "expertXAS" / TASK / "checkpoints" / "best*.ckpt",
    ]
    hits = []
    for pattern in patterns:
        hits.extend(glob.glob(str(pattern)))
    if not hits:
        raise FileNotFoundError(f"No ExpertXAS checkpoint found for {TASK}")
    return Path(sorted(hits, key=os.path.getmtime)[-1])


def per_spectrum_mse(pred, target):
    return np.mean((target - pred) ** 2, axis=1)


# Old baseline: original frozen-M3GNet features + old ExpertXAS head.
old_X_test = np.loadtxt(DATA_DIR / f"{TASK}_test_X.txt", dtype=np.float32)
old_y_test = np.loadtxt(DATA_DIR / f"{TASK}_test_y.txt", dtype=np.float32)
old_expert = load_xasblock(latest_expert_ckpt(), input_dim=64)
old_pred = predict_array(old_expert, old_X_test)
old_mse = per_spectrum_mse(old_pred, old_y_test)

# Existing 64D D1 ensemble, if present.
d1_prediction_files = sorted((REPO_ROOT / "output" / "training" / "gnnXASStyleCuFEFF" / "d1_ensemble").glob("*/member_predictions.npz"))
pred64 = mse64 = None
if d1_prediction_files:
    d1_data = np.load(d1_prediction_files[-1], allow_pickle=True)
    pred64 = d1_data["preds_test"].mean(axis=0)
    mse64 = per_spectrum_mse(pred64, old_y_test)
else:
    print("No 64D D1 member predictions found; histogram will omit 64D ensemble.")

# Best 128D candidate by validation eta among singles, per-seed ensembles, and all-new-seeds ensemble.
preds128_val = np.stack(all_preds_val)
preds128_test = np.stack(all_preds_test)

candidates = []
for i, row in members_128d.iterrows():
    candidates.append((float(row["val_eta"]), row["name"], preds128_test[i]))

for seed, group in members_128d.groupby("encoder_seed"):
    idx = list(group.index)
    pred_val = preds128_val[idx].mean(axis=0)
    pred_test = preds128_test[idx].mean(axis=0)
    candidates.append((eta(pred_val, y_ref["val"], train_y_ref)["eta"], f"128D seed{seed} 4-head ensemble", pred_test))

all_idx = list(range(len(members_128d)))
pred_val = preds128_val[all_idx].mean(axis=0)
pred_test = preds128_test[all_idx].mean(axis=0)
candidates.append((eta(pred_val, y_ref["val"], train_y_ref)["eta"], "128D all-new-seeds ensemble", pred_test))

best128_val_eta, best128_label, pred128 = max(candidates, key=lambda item: item[0])
mse128 = per_spectrum_mse(pred128, old_y_test)

hist_items = [("old frozen-M3GNet ExpertXAS", old_pred, old_mse)]
if pred64 is not None:
    hist_items.append(("64D D1 ensemble", pred64, mse64))
hist_items.append((f"best 128D: {best128_label}", pred128, mse128))

hist_rows = []
for label, pred, mse in hist_items:
    metrics = eta(pred, old_y_test, train_y_ref)
    hist_rows.append({
        "model": label,
        "test_eta": metrics["eta"],
        "test_median_mse": metrics["median_mse"],
        "mean_mse": float(np.mean(mse)),
        "p90_mse": float(np.quantile(mse, 0.90)),
    })

hist_df = pd.DataFrame(hist_rows)
display(hist_df)
hist_df.to_csv(COMPARISON_DIR / "test_mse_histogram_summary.csv", index=False)

bins = 10 ** np.linspace(-4, 0, 24)
plt.figure(figsize=(7, 4.5))
for label, _pred, mse in hist_items:
    plt.hist(mse, bins=bins, alpha=0.45, label=label)
plt.xscale("log")
plt.xlabel("Per-spectrum test MSE")
plt.ylabel("Count")
plt.title("Cu_FEFF test-error distribution")
plt.legend(fontsize=8)
plt.tight_layout()
out_path = COMPARISON_DIR / "test_mse_histogram_old_64d_best128d.png"
plt.savefig(out_path, dpi=200)
plt.show()
print("saved histogram:", out_path)
